# OpenMythos R16 — 7B QLoRA Fine-Tuning

Fine-tune Qwen2.5-Coder-7B on governance SFT data using QLoRA 4-bit.

**Hardware:** Google Colab T4 GPU (15GB VRAM)  
**Time:** ~2-3 hours  
**Cost:** Free (Colab) or ~$0.50 (Colab Pro)

In [ ]:
# Install dependencies
!pip install -q unsloth transformers datasets trl accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 2048
DTYPE = None  # Auto-detect
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
# Load dataset
from datasets import load_dataset

# Upload r15-merged-sft.jsonl to Colab first
dataset = load_dataset("json", data_files="r15-merged-sft.jsonl", split="train")

def format_example(ex):
    return f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['output']}"

dataset = dataset.map(lambda x: {"text": format_example(x)})

In [ ]:
# Train
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=TrainingArguments(
        output_dir="./output",
        num_train_epochs=5,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        warmup_steps=20,
        logging_steps=10,
        save_strategy="steps",
        save_steps=50,
        fp16=True,
        optim="adamw_8bit",
        report_to="none",
    ),
)

trainer.train()

In [ ]:
# Save model
model.save_pretrained("./openmythos-r16-7b")
tokenizer.save_pretrained("./openmythos-r16-7b")

# Convert to GGUF for Ollama
!pip install -q llama-cpp-python
!python -m llama_cpp.convert ./openmythos-r16-7b --outtype q4_k_m --outfile openmythos-r16-7b-q4.gguf

# Download
from google.colab import files
files.download("openmythos-r16-7b-q4.gguf")